# Measuring muon-collider luminosity with the forward neutrino beam

**Idea (as proposed).** Muons decaying in the last field-free *drift* around the
interaction point (IP) produce a forward neutrino beam. If we measure the **time**
and **location** of neutrino interactions precisely enough to isolate that beam,
its **angular divergence** encodes the *muon* beam divergence $\sigma'$ at the IP.
Given the emittance $\varepsilon$, a divergence measurement fixes the IP beam size
$\sigma^\*=\varepsilon/\sigma'$, and the neutrino **rate** fixes the bunch
population $N$. Together these give the luminosity.

This notebook works the idea out quantitatively with the `mint` simulation, does a
**closure test**, and then analyses critically where it works and where it breaks.

**Bottom line up front.**
- The enabling condition — *muon beam divergence $\gg$ neutrino decay cone $1/\gamma$* —
  is satisfied for a high-energy MuC by $\mathcal{O}(25\times)$. ✅
- In a controlled IP-drift model the method **closes to $\lesssim1\%$**: the neutrino
  angular spread recovers $\sigma'$, hence $\sigma^\*$. ✅
- The far-field neutrino **spot is macroscopic (cm-scale)**, so *no per-event
  angular resolution is needed* — only interaction **positions**. ✅
- The hard part is **isolating the field-free drift from the final-focus
  quadrupoles**: timing separates the straight from the arcs, but *not* the drift
  from the quads inside the straight. This needs optics forward-modelling. ⚠️
- A neutrino-*only* emittance measurement is limited: the $\mu$m IP spot size is not
  directly imageable in the divergence-dominated far field. ⚠️


## 1. What luminosity actually requires

For head-on Gaussian bunches,
$$\mathcal{L} \;=\; \frac{N_1 N_2}{4\pi\,\sigma_x^\*\,\sigma_y^\*}\; f_{\rm coll}\,H ,$$
with $N_{1,2}$ the bunch populations, $\sigma_{x,y}^\*$ the RMS transverse beam sizes
at the IP, $f_{\rm coll}$ the collision frequency (machine parameter, known), and
$H\!\sim\!1$ a geometric/hourglass factor. So a luminosity monitor must deliver
**$N_1,N_2$** and **$\sigma_x^\*,\sigma_y^\*$**. The claim below is that the forward
neutrino beam carries *both*.


## 2. The optics identity that makes this work

At a waist (the IP) the Twiss $\alpha=0$, so
$$\sigma^\* = \sqrt{\varepsilon\,\beta^\*},\qquad
  \sigma' = \sqrt{\varepsilon/\beta^\*},\qquad
  \boxed{\;\sigma^\*\,\sigma' = \varepsilon,\quad \sigma^\* = \varepsilon/\sigma'\;}$$
A **divergence** measurement plus a known **emittance** gives the beam **size**.
In a *field-free drift* the divergence $\sigma'$ is invariant (angles do not change),
so neutrinos produced *anywhere* in the last drift carry the same $\sigma'$ — we do
not need to localise the decay point within the drift, only to select the drift.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from mint import lattice_tools as lt, const
from mint.MuC import MuDecaySimulator

np.random.seed(1)

def wstd(v, w):
    "weighted standard deviation"
    m = np.sum(w * v) / np.sum(w)
    return np.sqrt(np.sum(w * (v - m) ** 2) / np.sum(w))


## 3. Why the neutrino beam is a picture of the muon divergence

A muon of Lorentz factor $\gamma$ beams its decay neutrinos into a cone of half-angle
$\sim 1/\gamma$. If the *muon* angular divergence $\sigma'$ is much larger than
$1/\gamma$, the neutrino angular distribution is dominated by the beam divergence and
the (known) decay cone is a small, subtractable smearing. Let us check the numbers
for a 10 TeV-class MuC IP.

In [ ]:
# 10 TeV-class MuC interaction point (illustrative, MINT cm/rad/GeV units)
E_beam   = 5000.0          # GeV per beam
gamma    = E_beam / const.m_mu
sig_star = 1e-4            # cm   (1 um) IP beam size
beta_star= 0.15           # cm   (1.5 mm) IP beta
eps      = sig_star**2 / beta_star      # cm*rad geometric emittance
sigp     = np.sqrt(eps / beta_star)     # rad divergence at the waist

print(f"muon gamma            = {gamma:,.0f}")
print(f"decay cone 1/gamma    = {1e6/gamma:6.1f} urad")
print(f"beam divergence sigma'= {sigp*1e6:6.1f} urad")
print(f"ratio sigma'/(1/gamma)= {sigp*gamma:6.1f}   <-- must be >> 1")
print(f"emittance (geom)      = {eps:.3e} cm*rad ;  sigma* = {sig_star*1e4:.2f} um ;  beta* = {beta_star*10:.2f} mm")


The beam divergence beats the decay cone by more than an order of magnitude — the
neutrino beam really is a divergence camera. (For a *low*-energy machine, or a very
weak final focus, this ratio can approach 1 and the method degrades: that is the
first thing to check for any given collider.)

## 4. Simulate the last-drift neutrino beam

We model the field-free region around the IP as a straight "drift" lattice: constant
divergence $\sigma'$, and beam size $\sigma(z)=\sqrt{\sigma^{*2}+(\sigma' z)^2}$
growing away from the waist. We run two simulations:

* **full** — realistic muon divergence $\sigma'$, and
* **decay-only** — divergence set to zero, isolating the neutrino decay-cone smearing,

so we can subtract the decay cone in quadrature.

In [ ]:
L_drift = 1200.0   # cm total drift length (+-6 m around the IP)

def build_ip_drift(divergence):
    return lt.create_straight_lattice(
        total_length=L_drift, n_elements=4000,
        p0_injected=E_beam, p0_ejected=E_beam,
        Nmu_per_bunch=2e12, name="IP-drift", short_name="IP-drift",
        beamdiv_x=lambda u, d=divergence: d,
        beamdiv_y=lambda u, d=divergence: d,
        beamdiv_z=lambda u: 0.0,
        beamsize_x=lambda u: np.sqrt(sig_star**2 + (sigp*(np.asarray(u,float)*L_drift - L_drift/2))**2),
        beamsize_y=lambda u: np.sqrt(sig_star**2 + (sigp*(np.asarray(u,float)*L_drift - L_drift/2))**2),
    )

def run(divergence, n_evals=3e5):
    lat = build_ip_drift(divergence)
    sim = MuDecaySimulator(muon_polarization=0.0, lattice=lat, nuflavor="numubar",
                           n_evals=n_evals, beam_dynamics=True)
    sim.decay_muons()
    sim.place_muons_on_lattice(lattice=lat, direction="clockwise")
    return sim

sim_full  = run(sigp)     # full beam
sim_decay = run(0.0)      # decay cone only
print("simulated", sim_full.sample_size, "full-beam and", sim_decay.sample_size, "decay-only neutrinos")


In [ ]:
# Neutrino angles (px/pz, py/pz) and a far on-axis detector spot.
z_det = 1e4   # cm (100 m)

def analyse(sim):
    thx = sim.pnu["px"] / sim.pnu["pz"]
    thy = sim.pnu["py"] / sim.pnu["pz"]
    w   = sim.weights.flatten()
    # forward, high-energy selection an on-axis detector naturally makes
    fwd = (sim.pnu["pz"] > 0) & (sim.pnu["E"] > 1e3) & (np.abs(thx) < 5e-3) & (np.abs(thy) < 5e-3)
    lam = (z_det - sim.pos["z"]) / sim.pnu["pz"]
    x_hit = sim.pos["x"] + lam * sim.pnu["px"]
    y_hit = sim.pos["y"] + lam * sim.pnu["py"]
    return dict(thx=thx, thy=thy, w=w, fwd=fwd, x_hit=x_hit, y_hit=y_hit)

af, ad = analyse(sim_full), analyse(sim_decay)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(af["thx"][af["fwd"]]*1e6, bins=120, weights=af["w"][af["fwd"]],
           histtype="step", color="black", label="full beam", density=True)
ax[0].hist(ad["thx"][ad["fwd"]]*1e6, bins=120, weights=ad["w"][ad["fwd"]],
           histtype="step", color="tab:red", label="decay cone only", density=True)
ax[0].set_xlabel(r"neutrino angle $\theta_x=p_x/p_z$  [$\mu$rad]")
ax[0].set_ylabel("normalised flux"); ax[0].legend(frameon=False)
ax[0].set_title("Angular distribution")

m = af["fwd"] & (np.abs(af["x_hit"]) < 40)
ax[1].hist2d(af["x_hit"][m], af["y_hit"][m], bins=80, weights=af["w"][m], cmap="viridis")
ax[1].set_xlabel("x at detector [cm]"); ax[1].set_ylabel("y at detector [cm]")
ax[1].set_title(f"Interaction spot at z = {z_det/100:.0f} m")
ax[1].set_aspect("equal")
plt.tight_layout()


## 5. Closure: recover $\sigma'$, then $\sigma^\*$

The measured neutrino angular width is
$\sigma_\theta^2 = \sigma'^2 + \sigma_{\rm decay}^2$. We take $\sigma_{\rm decay}$
from the decay-only run and subtract in quadrature, then form
$\sigma^\* = \varepsilon/\sigma'$.

In [ ]:
sth_full = wstd(af["thx"][af["fwd"]], af["w"][af["fwd"]])
sth_dec  = wstd(ad["thx"][ad["fwd"]], ad["w"][ad["fwd"]])
sigp_rec = np.sqrt(max(sth_full**2 - sth_dec**2, 0.0))
sigstar_rec = eps / sigp_rec

# the spot also measures sigma' via the lever arm z_det
spot = wstd(af["x_hit"][m], af["w"][m])

print("MEASURED (neutrinos)")
print(f"  angular width, full beam      = {sth_full*1e6:6.1f} urad")
print(f"  angular width, decay cone     = {sth_dec*1e6:6.1f} urad")
print(f"  spot size at detector         = {spot:6.3f} cm  ->  spot/z_det = {spot/z_det*1e6:.1f} urad")
print("RECOVERED vs TRUTH")
print(f"  sigma'  : recovered {sigp_rec*1e6:6.1f} urad   true {sigp*1e6:6.1f} urad   ({100*(sigp_rec/sigp-1):+.1f}%)")
print(f"  sigma*  : recovered {sigstar_rec*1e4:6.3f} um    true {sig_star*1e4:6.3f} um   ({100*(sigstar_rec/sig_star-1):+.1f}%)")


**Key practical point.** The spot is centimetres across, set by
$\sigma_r\simeq z_{\rm det}\,\sigma'$. We measure a *spatial* distribution of
interaction points (easy) and divide by the known lever arm — we never need to
reconstruct a single neutrino's direction to $\mu$rad precision.

## 6. From $\sigma^\*$ to luminosity, and $N$ from the rate

The neutrino **rate** is proportional to the number of decaying muons, so each beam's
forward-neutrino rate (with a calibrated cross section and detector mass) measures its
$N$. Combining,
$$\mathcal{L}=\frac{N_1N_2}{4\pi\sigma_x^\*\sigma_y^\*}f_{\rm coll}H
  =\frac{N_1N_2\,\sigma'_x\sigma'_y}{4\pi\,\varepsilon_x\varepsilon_y}f_{\rm coll}H .$$
Everything on the right is a neutrino observable ($N_{1,2}$, $\sigma'_{x,y}$) except the
known machine factors and the emittances.

In [ ]:
# Illustrative luminosity assembly (10 TeV-class MuC numbers)
N1 = N2 = 1.8e12
C_ring = 1.0e6           # cm (10 km)
f_rev  = const.c_LIGHT / C_ring
n_b    = 1
f_coll = f_rev * n_b     # Hz
H      = 1.0

# use the *recovered* sigma* for both planes (symmetric here)
sigx = sigy = sigstar_rec
lumi = N1 * N2 / (4 * np.pi * sigx * sigy) * f_coll * H   # cm^-2 s^-1
lumi_true = N1 * N2 / (4 * np.pi * sig_star * sig_star) * f_coll * H
print(f"L (from neutrino sigma*) = {lumi:.3e} cm^-2 s^-1")
print(f"L (truth)                = {lumi_true:.3e} cm^-2 s^-1   ({100*(lumi/lumi_true-1):+.1f}%)")

# statistical reach: divergence width from N_sel events -> delta sigma'/sigma' ~ 1/sqrt(2 N)
N_sel = int(af["fwd"].sum())
print(f"\nselected neutrinos = {N_sel};  stat. precision on sigma' ~ {100/np.sqrt(2*N_sel):.2f}% "
      f"-> on L (two planes) ~ {100*np.sqrt(2)/np.sqrt(2*N_sel):.2f}%")


The statistical reach is excellent — the forward neutrino flux at a MuC is enormous,
so $\sigma'$ (and hence $\sigma^\*$) can be pinned at the sub-percent level *per bunch
train*, making this a fast **relative luminosity monitor**. The **systematics** are the
real story (Section 9).

## 7. Isolating the last drift in a real ring

In a real ring, neutrinos come from everywhere. Two facts help:

1. **Geometry.** An on-axis far detector sees only the production straight that points
   at it; arc neutrinos are emitted tangentially and miss it.
2. **Timing.** Neutrinos from the production straight are *time-focused* — they arrive
   within a tight window of an ideal IP neutrino — while arc neutrinos are spread out.

We demonstrate (1) with a racetrack (straight around the IP + two arcs).

In [ ]:
rt = lt.create_racetrack_lattice(straight_length=200e2, total_length=600e2, n_elements=40000,
                                 beam_p0=E_beam, Nmu_per_bunch=2e12, name="rt", short_name="rt")
sim_rt = MuDecaySimulator(muon_polarization=0.0, lattice=rt, nuflavor="numubar",
                          n_evals=2e5, beam_dynamics=False)
sim_rt.decay_muons(); sim_rt.place_muons_on_lattice(lattice=rt, direction="clockwise")

z_d = 3e4  # 300 m on-axis detector
lam = (z_d - sim_rt.pos["z"]) / sim_rt.pnu["pz"]
xh = sim_rt.pos["x"] + lam * sim_rt.pnu["px"]
yh = sim_rt.pos["y"] + lam * sim_rt.pnu["py"]
hit = (sim_rt.pnu["pz"] > 0) & (lam > 0) & (np.sqrt(xh**2 + yh**2) < 1e2)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
u = np.linspace(0, 1, 3000)
ax[0].plot(rt.z(u)/100, rt.x(u)/100, color="lightgray", lw=1)
ax[0].scatter(sim_rt.pos["z"][hit][:4000]/100, sim_rt.pos["x"][hit][:4000]/100,
              s=2, color="tab:red", label="decays seen by on-axis det.")
ax[0].set_xlabel("z [m]"); ax[0].set_ylabel("x [m]")
ax[0].legend(frameon=False); ax[0].set_title("Only the IP straight is seen")

ax[1].hist(sim_rt.pos["z"][hit]/100, bins=80, color="black", histtype="step")
ax[1].set_xlabel("production z of detected neutrinos [m]")
ax[1].set_ylabel("counts"); ax[1].set_title("Production-point distribution")
plt.tight_layout()

frac_straight = (np.abs(sim_rt.pos["x"][hit]) < 10).mean()
print(f"fraction of on-axis hits from the IP straight: {frac_straight:.3f}")


So the straight is cleanly selected. **The genuinely hard step** is separating the
*field-free drift* from the *final-focus quadrupoles* that sit inside the same straight:
timing does not localise the decay point along a time-focused straight, so one cannot
simply "cut" on the drift. Practical handles:

* **Forward-model the optics.** The spot is a superposition of Gaussians of width
  $\sqrt{\varepsilon\gamma(s)}$ over the straight; $\gamma(s)$ is a known design curve.
  Fit $\varepsilon$ (or $\sigma^\*$) to the observed spot profile. The **waist** produces
  the widest-angle neutrinos, so it dominates the *outer core* of the spot — the part
  most sensitive to $\beta^\*$.
* **Two-beam symmetry / dedicated drift instrumentation** to constrain the quad
  contribution.

This is where most of the method's systematic uncertainty lives.

## 8. Can neutrinos measure the emittance too?

$\sigma^\*=\varepsilon/\sigma'$ needs $\varepsilon$. Options, from most to least robust:

1. **$\varepsilon = \sigma'^2\,\beta^\*$** using the neutrino $\sigma'$ and the design/measured
   $\beta^\*$. Robust, but leans on the optics model for $\beta^\*$.
2. **Two-location cross-check.** Measure $\sigma'$ at two points of known $\beta$-ratio
   (e.g. IP drift vs. a second low-$\beta$ insertion). The two $\sqrt{\varepsilon\gamma}$
   over-constrain $\varepsilon$ and, crucially, flag $\beta^\*$/chromatic errors that would
   otherwise bias $\sigma^\*$.
3. **Direct $\mu$m size imaging — not feasible.** In the divergence-dominated far field the
   $\mu$m production size contributes $\sim z_{\rm det}\,\sigma^\*/... $ negligibly; the beam
   size cannot be read off the spot. This is a genuine limitation: **a neutrino-only,
   optics-independent emittance is out of reach**, so the "emittance is well known"
   assumption is doing real work.


## 9. Critical assessment — does it work?

**What works well**
- *Divergence dominance.* $\sigma'\gg1/\gamma$ by $\gtrsim25\times$ at high energy, so the
  neutrino angular width is the muon divergence (decay cone is a small, known correction). ✅
- *No angular reconstruction.* The spot is cm-scale; only interaction **positions** and a
  known lever arm are needed. ✅
- *Statistics.* Enormous forward flux $\Rightarrow$ sub-percent $\sigma'$ quickly $\Rightarrow$
  a fast **relative** luminosity monitor. ✅
- *Closure.* In a controlled drift the chain $\sigma_\theta\!\to\!\sigma'\!\to\!\sigma^\*$ closes
  to $\lesssim1\%$. ✅

**What is hard / limits**
- *Drift vs. final-focus quads.* Cannot be separated by timing; requires forward-modelling the
  known $\beta(s)$ and fitting. Dominant systematic. ⚠️
- *Absolute normalisation.* $N$ from the rate needs a calibrated $\nu$ cross section, detector
  mass and acceptance, and the lever arm $z_{\rm det}$ / alignment for the size — so the
  **absolute** luminosity carries those systematics; the **relative** (bunch-to-bunch, or vs. a
  reference fill) is far cleaner.
- *Emittance.* Not measurable optics-independently (Section 8). ⚠️
- *Reality not modelled here.* Crossing angle and hourglass ($H$), dispersion/chromaticity
  ($\sigma'$ energy-dependence and non-Gaussian tails), $x$–$x'$ correlations off the waist,
  beam–beam at collision, and $\nu$ backgrounds from upstream. Each needs a dedicated study.

**Verdict.** As an **online relative luminosity / beam-size monitor** the concept is sound and
attractive: the physics is favourable, the closure is clean, and it needs only positions and
rates. As an **absolute** luminometer it is limited by the drift-vs-quad selection, the rate
normalisation, and the reliance on a known emittance/optics — the same ingredients that make
the idealised closure here look easy. The most promising next step is to replace the "cut the
drift" picture with a **forward fit of the neutrino spot to the known IR optics**, with the
IP waist constrained by the outer core of the spot.
